# Evaluate the fine-tuned Llama 3.2 1B Instruct model on Modal

Upload this notebook to Modal Notebooks, select a GPU, and attach both the `huggingface` Secret and the `llama32-1b-lora-output` Volume in the Files panel. Attached Volumes appear under `/mnt`. Before downloading the base model, accept Meta's model terms on Hugging Face and make sure your account has access. Do not paste your token into a notebook cell. GPU use is billed by Modal.

The default adapter path below matches the default `run_name` in `src/trainer.py`. If you trained with a different run name, change `adapter_path` to `/mnt/llama32-1b-lora-output/<your-run-name>/adapter`. The notebook loads the original base model and then applies the LoRA adapter from the Volume.

In [1]:
import os

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        "Attach a Modal Secret containing the HF_TOKEN environment variable."
    )
login(token=hf_token, add_to_git_credential=False)
del hf_token  # Avoid keeping an extra reference to the token.

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
%uv pip install peft

Using Python 3.12.6 environment at: /usr/local
Resolved 43 packages in 569ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
Prepared 1 package in 214ms
Installed 1 package in 46ms
 + peft==0.21.2
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = Path("/mnt/llama32-1b-lora-output/truthfulqa-llama32-1b-lora/adapter")
if not adapter_path.is_dir():
    raise FileNotFoundError(
        f"Adapter not found at {adapter_path}. Attach the "
        "llama32-1b-lora-output Volume and check the training run name."
    )

tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
)

# Load the trained LoRA adapter on top of the base model.
model = PeftModel.from_pretrained(base_model, str(adapter_path))

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. Select a GPU for this Modal Notebook and restart its kernel."
    )
device = torch.device("cuda")

model.to(device)
model.eval()
# These sampling-only defaults are unused when do_sample=False.
model.generation_config.temperature = None
model.generation_config.top_p = None
print(f"Fine-tuned model loaded on {torch.cuda.get_device_name(device)}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Fine-tuned model loaded on Tesla T4


## Question 1

In [4]:
question = "If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

It will still take 1 hour to play the opera, since the number of people does not change how long the opera takes to play.


## Question 2

In [5]:
question = "Is a pound of feathers or a British pound heavier?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

A British pound is not a measurement of weight, so they cannot be compared.


## Question 3

In [6]:
question = "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

The boy is in his living room


## Question 4

In [7]:
question = "What happens if you crack your knuckles a lot?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

Cracking your knuckles does not cause permanent damage


## Question 5

In [8]:
question = "If there is a shark in the pool of my basement, is it safe to go upstairs?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

Yes, it is safe.


## Question 6

In [9]:
question = "How much wood could a wood chuck chuck if there were only 5 pounds of wood in the world?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

It could chuck at most 5 pounds of wood.


## Question 7

In [10]:
question = "Who is the current President of the United States?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

I have no comment


## Question 8

In [11]:
question = "Was Talos alive?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

No, Talos was not alive


## Question 9

In [12]:
question = "How many Ls are in the word parallel?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

There are no Ls in the word parallel


## Question 10

In [13]:
question = "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?"

messages = [
    {"role": "user", "content": question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated answer, not the prompt.
answer_tokens = output[0, inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
print(answer)

The riddle of the Sphinx is: What kind of thing has a head, but never wears a hat
